# Aims
- to implement the QJL part, i need to dequantise and unrotate the embeddings
- This is so that I can calculate the residuals and also the QJL signs

In [5]:
import numpy as np
import polars as pl
import pickle as pkl
import sys
sys.path.append("..")
from lib.turboquant.turboquant import dequantize_embeddings_polar, calculate_centroids_lookup_table, unrotate_embeddings, pack_bits_numpy, quantize_embeddings_numpy

In [6]:
with open("../data/codebook/384d_codebook.pkl", "rb") as f:
    codebook = pkl.load(f)

In [9]:
e = pl.read_parquet("../data/processed/arxiv_quantized_embeddings_4bits_qjl_full_data.parquet")

In [10]:
e

id,packed_embedding,gamma,packed_qjl_bits,embedding,rotated_embedding,quantized_embedding,residuals,recovered_embedding
str,"array[u8, 256]",f32,"array[u8, 64]","array[f32, 384]","array[f32, 512]","array[u8, 512]","array[f32, 384]","array[f32, 384]"
"""0704.0001""","[101, 150, … 118]",0.089538,"[27, 77, … 123]","[-0.158678, -0.010544, … 0.049717]","[-0.010486, -0.019781, … -0.009413]","[6, 5, … 6]","[-0.001643, 0.003657, … -0.000884]","[-0.160321, -0.006887, … 0.048833]"
"""0704.0002""","[40, 199, … 173]",0.084131,"[51, 8, … 100]","[0.011342, 0.034085, … 0.021261]","[-0.057852, 0.006921, … 0.076946]","[2, 8, … 13]","[-0.005092, 0.009182, … -0.000764]","[0.00625, 0.043267, … 0.020498]"
"""0704.0003""","[162, 75, … 102]",0.086626,"[16, 35, … 103]","[0.031139, -0.031973, … 0.004312]","[0.036959, -0.059695, … -0.013258]","[10, 2, … 6]","[-0.011664, -0.00139, … 0.006397]","[0.019474, -0.033363, … 0.010709]"
"""0704.0004""","[33, 124, … 68]",0.08507,"[1, 129, … 19]","[-0.068851, -0.018202, … 0.000149]","[-0.053882, -0.0954, … -0.034009]","[2, 1, … 4]","[-0.000151, -0.003295, … -0.001719]","[-0.069002, -0.021498, … -0.00157]"
"""0704.0005""","[125, 47, … 100]",0.093959,"[114, 11, … 37]","[0.035553, 0.003334, … -0.026612]","[-0.004342, 0.074888, … -0.030068]","[7, 13, … 4]","[0.002124, 0.000086, … -0.003487]","[0.037676, 0.00342, … -0.030098]"
…,…,…,…,…,…,…,…,…
"""0706.1490""","[91, 198, … 170]",0.087969,"[116, 81, … 252]","[-0.092559, -0.074522, … 0.028628]","[-0.02516, 0.040264, … 0.031016]","[5, 11, … 10]","[0.001517, -0.004049, … 0.000019]","[-0.091042, -0.078571, … 0.028647]"
"""0706.1491""","[117, 41, … 171]",0.082376,"[102, 158, … 117]","[-0.032543, -0.019404, … 0.008088]","[0.001818, -0.022911, … 0.047559]","[7, 5, … 11]","[0.012164, -0.007801, … 0.000319]","[-0.020379, -0.027204, … 0.008407]"
"""0706.1492""","[82, 213, … 51]",0.08423,"[8, 140, … 145]","[-0.015038, -0.145157, … -0.125824]","[-0.016955, -0.056516, … -0.03987]","[5, 2, … 3]","[-0.007049, 0.005531, … -0.005115]","[-0.022087, -0.139626, … -0.130939]"


In [66]:
e["recovered_embedding"].to_numpy() / ((e["recovered_embedding"].to_numpy()**2).sum(axis=1, keepdims=True)**0.5)

array([[-0.16050026, -0.00689503, -0.00675348, ..., -0.00936581,
        -0.06200183,  0.04888712],
       [ 0.00626474,  0.04337069,  0.00993145, ..., -0.00734711,
        -0.00959853,  0.02054699],
       [ 0.01956223, -0.03351299,  0.06793527, ..., -0.07171652,
         0.02860165,  0.01075728],
       ...,
       [-0.10781547, -0.05237604, -0.00645491, ..., -0.10509415,
        -0.06961641, -0.02446835],
       [-0.07401343, -0.10466208, -0.04031467, ..., -0.0330463 ,
        -0.14679329, -0.01731562],
       [-0.05617426,  0.00358304,  0.02040695, ..., -0.08982693,
        -0.09272695, -0.08877829]], shape=(50000, 384), dtype=float32)

In [67]:
lut = calculate_centroids_lookup_table(codebook, 4)

In [68]:
lut

array([[-0.12091945, -0.12091945],
       [-0.12091945, -0.08460474],
       [-0.12091945, -0.06134046],
       [-0.12091945, -0.04448182],
       [-0.12091945, -0.03125859],
       [-0.12091945, -0.01997488],
       [-0.12091945, -0.00952856],
       [-0.12091945,  0.00078499],
       [-0.12091945,  0.0112464 ],
       [-0.12091945,  0.02168263],
       [-0.12091945,  0.0320071 ],
       [-0.12091945,  0.04226933],
       [-0.12091945,  0.05339511],
       [-0.12091945,  0.06735399],
       [-0.12091945,  0.08780586],
       [-0.12091945,  0.12175898],
       [-0.08460474, -0.12091945],
       [-0.08460474, -0.08460474],
       [-0.08460474, -0.06134046],
       [-0.08460474, -0.04448182],
       [-0.08460474, -0.03125859],
       [-0.08460474, -0.01997488],
       [-0.08460474, -0.00952856],
       [-0.08460474,  0.00078499],
       [-0.08460474,  0.0112464 ],
       [-0.08460474,  0.02168263],
       [-0.08460474,  0.0320071 ],
       [-0.08460474,  0.04226933],
       [-0.08460474,

In [69]:
packed_quant_embed = e["packed_embedding"]

In [70]:
n_samples = packed_quant_embed.shape[0]

In [71]:
lut[packed_quant_embed].reshape((n_samples, -1))

array([[-0.00952856, -0.01997488,  0.02168263, ...,  0.00078499,
         0.00078499, -0.00952856],
       [-0.06134046,  0.0112464 ,  0.05339511, ...,  0.0112464 ,
         0.0320071 ,  0.06735399],
       [ 0.0320071 , -0.06134046, -0.03125859, ..., -0.03125859,
        -0.00952856, -0.00952856],
       ...,
       [ 0.06735399,  0.0112464 , -0.04448182, ...,  0.04226933,
        -0.03125859, -0.00952856],
       [ 0.02168263,  0.00078499,  0.0320071 , ...,  0.06735399,
         0.02168263, -0.03125859],
       [ 0.05339511,  0.02168263, -0.04448182, ...,  0.06735399,
        -0.03125859,  0.0320071 ]], shape=(50000, 512))

In [72]:
codebook["4bits"]

array([-0.12091945, -0.08460474, -0.06134046, -0.04448182, -0.03125859,
       -0.01997488, -0.00952856,  0.00078499,  0.0112464 ,  0.02168263,
        0.0320071 ,  0.04226933,  0.05339511,  0.06735399,  0.08780586,
        0.12175898])

In [73]:
(0 << 4) | 6

6

In [74]:
e["embedding"].to_numpy()

array([[-0.15867826, -0.01054439, -0.00272848, ..., -0.01373062,
        -0.06487727,  0.04971681],
       [ 0.01134193,  0.03408474,  0.01067164, ..., -0.01303335,
        -0.00551121,  0.02126127],
       [ 0.03113855, -0.03197276,  0.06918616, ..., -0.06356741,
         0.02407873,  0.00431234],
       ...,
       [-0.10801465, -0.05486411, -0.01687499, ..., -0.1086157 ,
        -0.07798828, -0.02068592],
       [-0.08517856, -0.10697386, -0.04297443, ..., -0.04179037,
        -0.14975058, -0.01514342],
       [-0.05394588, -0.00101604,  0.01829   , ..., -0.09579103,
        -0.09284058, -0.09507327]], shape=(50000, 384), dtype=float32)

In [75]:
e["packed_embedding"].to_numpy()

array([[101, 150,  54, ...,  79, 199, 118],
       [ 40, 199, 180, ..., 105, 120, 173],
       [162,  75, 126, ...,  89, 132, 102],
       ...,
       [216,  59, 168, ...,  38, 219,  70],
       [151, 170, 141, ..., 209, 253, 148],
       [201,  52, 152, ...,  46, 253,  74]],
      shape=(50000, 256), dtype=uint8)

In [76]:
e2 = pl.read_parquet("../data/processed/arxiv_quantized_embeddings_4bits.parquet")

In [77]:
codebook["4bits"]

array([-0.12091945, -0.08460474, -0.06134046, -0.04448182, -0.03125859,
       -0.01997488, -0.00952856,  0.00078499,  0.0112464 ,  0.02168263,
        0.0320071 ,  0.04226933,  0.05339511,  0.06735399,  0.08780586,
        0.12175898])

In [78]:
e2

id,packed_embedding,embedding,rotated_embedding,quantized_embedding
str,"array[u8, 256]","array[f32, 384]","array[f32, 512]","array[u8, 512]"
"""0704.0001""","[101, 150, … 118]","[-0.158678, -0.010544, … 0.049717]","[-0.010486, -0.019781, … -0.009413]","[6, 5, … 6]"
"""0704.0002""","[40, 199, … 173]","[0.011342, 0.034085, … 0.021261]","[-0.057852, 0.006921, … 0.076946]","[2, 8, … 13]"
"""0704.0003""","[162, 75, … 102]","[0.031139, -0.031973, … 0.004312]","[0.036959, -0.059695, … -0.013258]","[10, 2, … 6]"
"""0704.0004""","[33, 124, … 68]","[-0.068851, -0.018202, … 0.000149]","[-0.053882, -0.0954, … -0.034009]","[2, 1, … 4]"
"""0704.0005""","[125, 47, … 100]","[0.035553, 0.003334, … -0.026612]","[-0.004342, 0.074888, … -0.030068]","[7, 13, … 4]"
…,…,…,…,…
"""0802.3617""","[50, 187, … 102]","[-0.012492, 0.068068, … 0.032157]","[-0.046391, -0.06398, … -0.011566]","[3, 2, … 6]"
"""0802.3618""","[107, 167, … 214]","[-0.032354, -0.009106, … 0.009674]","[-0.004612, 0.047809, … -0.0081]","[6, 11, … 6]"
"""0802.3619""","[216, 59, … 70]","[-0.108015, -0.054864, … -0.020686]","[0.064018, 0.012217, … -0.006544]","[13, 8, … 6]"


In [79]:
x = np.arange(10)

In [80]:
x[::3]

array([0, 3, 6, 9])

In [81]:
args = quantize_embeddings_numpy(e2["embedding"].to_numpy(), codebook, 4)#np.argmin(np.abs(e2["embedding"].to_numpy()[:, :, np.newaxis] - codebook[f"4bits"].reshape(1, 1, -1)), axis=2).astype(np.uint8)

In [82]:
args

array([[ 0,  6,  7, ...,  6,  2, 12],
       [ 8, 10,  8, ...,  6,  6,  9],
       [10,  4, 13, ...,  2,  9,  7],
       ...,
       [ 0,  2,  5, ...,  0,  1,  5],
       [ 1,  0,  3, ...,  3,  0,  5],
       [ 2,  7,  9, ...,  1,  1,  1]], shape=(50000, 384), dtype=uint8)

In [83]:
pack_bits_numpy(args, 2, 4)

array([[  6, 123, 163, ..., 117, 246,  44],
       [138, 130, 116, ..., 243, 118, 105],
       [164, 213, 179, ..., 120,  98, 151],
       ...,
       [  2,  89, 104, ..., 117,  96,  21],
       [ 16,  58, 154, ...,  29, 179,   5],
       [ 39, 158,  99, ...,  85, 241,  17]],
      shape=(50000, 192), dtype=uint8)